# 🧬 SPS Self-Specialization — Capability Registry Demo

**Research claim demonstrated:** State 0 exists → reproduces itself → AI specializes the copy → verification activates State 1 → State 1 is persisted, inspected, reloaded, and reused.

The generated capability is now a real research artifact: JSON metadata + Python source.

## 🔬 Research flow

```text
        STATE 0
IntegerMultiplication
             │
             ▼
         REPLICATE
             │
             ▼
           S0-C
             │
             ▼
    SPECIALIZE with Qwen
             │
             ▼
           VERIFY ✓
             │
             ▼
           STATE 1
     FloatMultiplication
             │
       ┌─────┴─────┐
       ▼           ▼
    JSON + .py   REUSE
```

## 1. Install Ollama first

Run this cell first. Ollama is installed and started from `/content`, so repository deletion/recloning cannot invalidate its working directory.

In [ ]:
%cd /content
!apt-get update -qq
!apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh
!pkill -9 ollama || true
!pkill -9 llama-server || true
!nohup ollama serve >/tmp/ollama.log 2>&1 &
!sleep 5
!ollama --version
!curl -sf http://127.0.0.1:11434/api/tags || (cat /tmp/ollama.log; exit 1)

# Pull the local coding model used by the prototype.
!ollama pull qwen2.5-coder:7b

## 2. Clone latest code and install dependencies

In [ ]:
%cd /content
!rm -rf self-specialization
!git clone -q https://github.com/muhammadnaumantahir/self-specialization.git
%cd /content/self-specialization
!pip -q install -r requirements.txt pytest
!echo 'Repository commit:'
!git rev-parse HEAD
!echo '
Available Ollama models:'
!ollama list

## 3. Verify the implementation

The suite validates replication, AI specialization, verification, registry integration, failure diagnostics, persistence, inspection, and reload.

In [ ]:
%cd /content/self-specialization
!PYTHONPATH=. pytest -q

## 4. Run the real SPS experiment

The demo creates a fresh persistent registry under `/tmp/sps-capability-registry` for every run. This keeps the first request demonstrably missing the float capability.

In [ ]:
%cd /content/self-specialization
import os
os.environ['OLLAMA_MODEL'] = 'qwen2.5-coder:7b'
!PYTHONPATH=. python experiments/self_specialization_demo.py

## 📋 What to show your supervisor

Look for these sections in the output:

- 🔵 **PHASE 1 — STATE 0**: `IntegerMultiplication` exists before the float request.
- 🟡 **PHASE 3 — NEW REQUEST**: `[float, float] → float` is missing.
- 🟣 **CAPABILITY CREATED AT RUNTIME**: shows **ID, Name, State, Parent, Created, Location, Storage, and Source**.
- 📜 **EVOLUTION EVENTS**: shows the runtime lineage and verification/activation events.
- 📋 **REGISTRY INSPECTION**: demonstrates `registry.get(...)`, `registry.list_active()`, and `registry.inspect(...)`.
- 🔁 **RELOAD AND REUSE**: proves the S1 capability can be reconstructed from persistent storage and executed without another specialization request.

### Persistent artifacts
```text
/tmp/sps-capability-registry/
├── registry.json                 ← registry index
├── records/<capability-id>.json ← capability metadata/events
└── sources/<id>_<name>.py       ← exact capability source
```

### Thesis interpretation
The important observation is not simply that Qwen writes Python. The system **detects a capability gap, reproduces an existing capability, transforms the reproduction into a new specialization, verifies it, registers it as a new state, persists it, and later retrieves it for reuse**.

## 🛠️ If Ollama fails

Run the diagnostic cell below. It checks the working directory, Ollama processes, API response, and server log.

In [ ]:
%cd /content
!pwd
!echo '
Ollama processes:'
!ps aux | grep -E 'ollama|llama-server' | grep -v grep || true
!echo '
Ollama API:'
!curl -s http://127.0.0.1:11434/api/tags || true
!echo '
Ollama log:'
!cat /tmp/ollama.log